# 01. Первинний аналіз Telco Customer Churn

Мета: зрозуміти структуру та якість даних, розподіл відтоку й потенційні особливості ознак. Це описовий аналіз, а не навчання чи оцінювання моделі. Оригінальний CSV не змінюється.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## Завантаження даних

Завантажте CSV згідно з README, якщо він ще відсутній. Ноутбук підтримує запуск із кореня проєкту або каталогу `notebooks`.

In [ ]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
data_path = project_root / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
if not data_path.is_file():
    raise FileNotFoundError(f"Не знайдено {data_path}. Інструкції із завантаження є в README.md.")

df = pd.read_csv(data_path)
print(f"Рядків: {df.shape[0]}, стовпців: {df.shape[1]}")
display(df.head())

## Структура та якість даних

Перевіряємо типи, пропущені значення, повні дублікати та повтори ідентифікаторів. `customerID` — ідентифікатор, а не предиктор.

In [ ]:
df.info()
display(pd.DataFrame({"тип": df.dtypes.astype(str), "пропуски": df.isna().sum(), "унікальних": df.nunique(dropna=True)}))
print(f"Повні дублікати рядків: {df.duplicated().sum()}")
print(f"Повторні customerID: {df['customerID'].duplicated().sum()}")
display(df.describe(include="all").T)

## Перевірка `TotalCharges`

Поле може містити порожні рядки. Діагностуємо порожні й інші нечислові значення окремо; перетворена серія існує лише в пам'яті і не змінює сирий CSV.

In [ ]:
total_text = df["TotalCharges"].astype("string").str.strip()
total_numeric = pd.to_numeric(total_text, errors="coerce")
empty_total = total_text.isna() | total_text.eq("").fillna(False)
invalid_total = total_numeric.isna() & ~empty_total
print(f"Порожні або пропущені TotalCharges: {empty_total.sum()}")
print(f"Інші нечислові TotalCharges: {invalid_total.sum()}")
display(df.loc[empty_total | invalid_total, ["customerID", "tenure", "TotalCharges"]])

## Цільова змінна

Порівняйте кількість і частку класів. Дисбаланс може зробити accuracy недостатньою метрикою.

In [ ]:
churn_counts = df["Churn"].value_counts(dropna=False)
display(pd.DataFrame({"кількість": churn_counts, "частка": churn_counts / len(df)}))
ax = sns.countplot(data=df, x="Churn", order=["No", "Yes"])
ax.set(title="Розподіл відтоку", xlabel="Churn", ylabel="Кількість клієнтів")
plt.show()

## Числові та категоріальні ознаки

Ці графіки показують асоціації у повному наборі, а не причинні зв'язки. Не використовуйте їх як незалежну тестову оцінку.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=df, x="tenure", hue="Churn", multiple="layer", bins=30, ax=axes[0])
sns.histplot(data=df, x="MonthlyCharges", hue="Churn", multiple="layer", bins=30, ax=axes[1])
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=df, x="Contract", hue="Churn", ax=axes[0])
sns.countplot(data=df, x="InternetService", hue="Churn", ax=axes[1])
fig.tight_layout()
plt.show()

## Висновки та наступні кроки

Для зафіксованого в README CSV початкові спостереження такі:

- 7 043 рядки, 21 стовпець; повних дублікатів і повторних `customerID` не виявлено.
- `Churn = Yes`: 1 869 (26,54%); `No`: 5 174 (73,46%). Класи незбалансовані, тому однієї accuracy недостатньо.
- `TotalCharges` містить 11 порожніх значень і жодного іншого нечислового значення; у всіх 11 записів `tenure = 0`. Це спостереження, а не підстава автоматично замінювати значення нулями.
- `customerID` є ідентифікатором; його не слід використовувати як ознаку моделі.

Це опис усього набору, а не оцінка моделі. Перед вибором ознак та розробкою моделі відокремте тестову вибірку; план експериментів наведено в README.